In [1]:
from datasets import load_dataset

dataset = load_dataset("Jr23xd23/ArabicText-Large", split="train")
dataset = dataset.shuffle(seed=42).select(range(10000))

c:\Users\computer\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import re

def clean_arabic(text):
    if text is None:
        return ""
    text = re.sub(r"[ـ]+", "", text)
    text = re.sub("[\u064B-\u065F]","",text)
    text = re.sub("[أإآٱ]","ا",text)
    text = re.sub("ة","ه",text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub("[^ا-ي0-9]+"," ",text).strip()

    return text.strip()

In [3]:
dataset = dataset.map(lambda x: {"text": clean_arabic(x["text"])})

In [4]:
def get_training_corpus():
    for i in range(0, len(dataset), 1000):
        batch = dataset[i:i+1000]["text"]
        yield [clean_arabic(t) for t in batch if t]

In [5]:
with open("wikitext-ar.txt", "w", encoding="utf-8") as f:
    for i in range(len(dataset)):
        f.write(dataset[i]["text"] + "\n")

In [6]:
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer,
)

tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))

In [7]:
tokenizer.normalizer = normalizers.Sequence(
    [normalizers.NFD()]
)

In [8]:
print(tokenizer.normalizer.normalize_str("بمنطقه الرياض في المملكه العربيه السعوديه"))

بمنطقه الرياض في المملكه العربيه السعوديه


In [9]:
pre_tokenizer = pre_tokenizers.WhitespaceSplit()
pre_tokenizer.pre_tokenize_str("بمنطقه الرياض في المملكه العربيه السعوديه")

[('بمنطقه', (0, 6)),
 ('الرياض', (7, 13)),
 ('في', (14, 16)),
 ('المملكه', (17, 24)),
 ('العربيه', (25, 32)),
 ('السعوديه', (33, 41))]

In [10]:
pre_tokenizer = pre_tokenizers.Sequence(
    [pre_tokenizers.WhitespaceSplit(), pre_tokenizers.Punctuation()]
)
pre_tokenizer.pre_tokenize_str("بمنطقه الرياض في المملكه العربيه السعوديه")

[('بمنطقه', (0, 6)),
 ('الرياض', (7, 13)),
 ('في', (14, 16)),
 ('المملكه', (17, 24)),
 ('العربيه', (25, 32)),
 ('السعوديه', (33, 41))]

In [11]:
special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
trainer = trainers.WordPieceTrainer(vocab_size=25000, special_tokens=special_tokens)

In [12]:
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)

In [13]:
tokenizer.model = models.WordPiece(unk_token="[UNK]")
tokenizer.train(["wikitext-ar.txt"], trainer=trainer)

In [14]:
encoding = tokenizer.encode("بمنطقه الرياض في المملكه العربيه السعوديه")
print(encoding.tokens)

['ب', '##منطقه ال', '##رياض', '## في الم', '##ملكه العربيه السعود', '##يه']


In [15]:
cls_token_id = tokenizer.token_to_id("[CLS]")
sep_token_id = tokenizer.token_to_id("[SEP]")
print(cls_token_id, sep_token_id)

2 3


In [16]:
tokenizer.post_processor = processors.TemplateProcessing(
    single=f"[CLS]:0 $A:0 [SEP]:0",
    pair=f"[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
    special_tokens=[("[CLS]", cls_token_id), ("[SEP]", sep_token_id)],
)

In [17]:
encoding = tokenizer.encode("بمنطقه الرياض في المملكه العربيه السعوديه")
print(encoding.tokens)

['[CLS]', 'ب', '##منطقه ال', '##رياض', '## في الم', '##ملكه العربيه السعود', '##يه', '[SEP]']


In [18]:
encoding = tokenizer.encode("وادي الدواسر", "بمنطقه الرياض في المملكه العربيه السعوديه")
print(encoding.tokens)
print(encoding.type_ids)

['[CLS]', 'و', '##ادي ', '##الد', '##واس', '##ر', '[SEP]', 'ب', '##منطقه ال', '##رياض', '## في الم', '##ملكه العربيه السعود', '##يه', '[SEP]']
[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]


In [19]:
tokenizer.decoder = decoders.WordPiece(prefix="##")

In [20]:
tokenizer.decode(encoding.ids)

'وادي الدواسر بمنطقه الرياض في المملكه العربيه السعوديه'

In [21]:
tokenizer.save("tokenizer.json")

In [22]:
new_tokenizer = Tokenizer.from_file("tokenizer.json")

In [23]:
from transformers import PreTrainedTokenizerFast

wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    # tokenizer_file="tokenizer.json", # You can load from the tokenizer file, alternatively
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
)